# Integrated Gradients — su BERTino Classifier

**Obiettivo:** capire *quali token* di una frase spingono il modello BERTino a classificarla come **complessa** (classe 1).

**Come funziona IG:**  
Si parte da una baseline neutra (tutti `[PAD]`) e si interpolano linearmente gli embedding verso quelli reali in `n_steps` passi. In ogni passo si calcola il gradiente del logit della classe "complessa" rispetto agli embedding. La somma di questi gradienti dà l'*attribuzione* di ogni token: quanto ha contribuito alla decisione finale.

- **attribuzione positiva (rosso/arancio)** → il token spinge verso "complesso"
- **attribuzione negativa (azzurro)** → il token spinge verso "semplice"

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import torch
import spacy
from difflib import SequenceMatcher
from IPython.display import display, HTML
from captum.attr import LayerIntegratedGradients
from captum.attr import visualization as viz
from transformers import AutoModelForSequenceClassification, AutoTokenizer

nlp = spacy.load("it_core_news_md")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

/data/miniconda3/envs/hlt/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## 2. Caricamento dati e costruzione eval_set

L'`eval_set` è una lista di dizionari, uno per frase del test set.  
Ogni dizionario contiene i token spaCy della frase originale e le silver label (`1` = token complesso secondo l'allineamento con la semplificazione).

In [10]:
test = pd.read_parquet("/code/HLTproject_code/data/test.parquet")

orig = (test[test["is_original"]]
        .drop_duplicates("original_sentence_idx")
        .set_index("original_sentence_idx")["text"])
simp = (test[~test["is_original"]]
        .drop_duplicates("original_sentence_idx")
        .set_index("original_sentence_idx")["text"])

pairs = (pd.concat([orig.rename("original"), simp.rename("simplification")], axis=1)
           .dropna().reset_index())
print(f"Coppie nel test set: {len(pairs)}")

Coppie nel test set: 7539


In [12]:
pairs.head(5)

,original_sentence_idx,original,simplification
0,10,"Prodotto dalla BBC, il film esce solo nel 1998...","Nel 1998, il film è stato rilasciato e ha rice..."
1,20,"ad aprile per la fiera del bestiame, a giugno ...","aprile per la fiera del bestiame, giugno per l..."
2,22,La popolarità e il prestigio di Marco Antonio ...,Questo fatto non ha influenzato la popolarità ...
3,42,Pur di riconquistare la ex ragazza Elaine ora ...,"Per riconquistare Elaine, Striker ha preso un ..."
4,80,"Cabrera firmò, all'età di 17 anni, con i New Y...","All'età di 17 anni, firmò con i New York Yankees"


In [13]:
def silver_labels_tipate(original, simplification):
    doc_o = [t for t in nlp(original) if not t.is_space]
    lem_o = [t.lemma_.lower() for t in doc_o]
    lem_s = [t.lemma_.lower() for t in nlp(simplification) if not t.is_space]
    tipo = ["keep"] * len(doc_o)
    for tag, i1, i2, j1, j2 in SequenceMatcher(a=lem_o, b=lem_s, autojunk=False).get_opcodes():
        if tag == "delete":
            for i in range(i1, i2): tipo[i] = "del"
        elif tag == "replace":
            for i in range(i1, i2): tipo[i] = "sub"
    tokens = [t.text for t in doc_o]
    labels = [int(x != "keep") for x in tipo]
    return tokens, labels, tipo

eval_set = []
for _, row in pairs.iterrows():
    doc_o = [t for t in nlp(row["original"]) if not t.is_space]
    tokens, labels, tipo = silver_labels_tipate(row["original"], row["simplification"])
    eval_set.append({
        "idx":        row["original_sentence_idx"],
        "original":   row["original"],
        "tokens":     tokens,
        "silver":     labels,
        "tipo":       tipo,
        "simp_tokens": [t.text for t in nlp(row["simplification"]) if not t.is_space],
    })

print(f"Eval set pronto: {len(eval_set)} frasi")

Eval set pronto: 7539 frasi


In [14]:
eval_df = pd.DataFrame(eval_set)
eval_df.head(5)

,idx,original,tokens,silver,tipo,simp_tokens
0,10,"Prodotto dalla BBC, il film esce solo nel 1998...","[Prodotto, dalla, BBC, ,, il, film, esce, solo...","[1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, ...","[sub, sub, sub, keep, keep, keep, sub, sub, su...","[Nel, 1998, ,, il, film, è, stato, rilasciato,..."
1,20,"ad aprile per la fiera del bestiame, a giugno ...","[ad, aprile, per, la, fiera, del, bestiame, ,,...","[1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, ...","[del, keep, keep, keep, keep, keep, keep, keep...","[aprile, per, la, fiera, del, bestiame, ,, giu..."
2,22,La popolarità e il prestigio di Marco Antonio ...,"[La, popolarità, e, il, prestigio, di, Marco, ...","[0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, ...","[keep, keep, keep, keep, sub, keep, keep, keep...","[Questo, fatto, non, ha, influenzato, la, popo..."
3,42,Pur di riconquistare la ex ragazza Elaine ora ...,"[Pur, di, riconquistare, la, ex, ragazza, Elai...","[1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, ...","[sub, sub, keep, del, del, del, keep, del, del...","[Per, riconquistare, Elaine, ,, Striker, ha, p..."
4,80,"Cabrera firmò, all'età di 17 anni, con i New Y...","[Cabrera, firmò, ,, all', età, di, 17, anni, ,...","[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...","[del, del, del, keep, keep, keep, keep, keep, ...","[All', età, di, 17, anni, ,, firmò, con, i, Ne..."


## Caricamento eval_set già salvato

In [26]:
import pickle

# eval_set salvato a monte 
with open("eval_set_full.pkl", "rb") as f:
    eval_set = pickle.load(f)

## 3. Caricamento del modello di classificazione

Usiamo il checkpoint con la LR migliore addestrato su Task 1A.

In [3]:
CKPT = "/code/HLTproject_code/task_1A_complexity_prediction/bert_cls/lr_5e-05/checkpoint-2828"

model = AutoModelForSequenceClassification.from_pretrained(CKPT)
tok   = AutoTokenizer.from_pretrained(CKPT)
model.eval().to(DEVICE)
print(f"Modello caricato su {DEVICE}")
print(f"Classi: {model.config.id2label}")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 8393.77it/s]


Modello caricato su cuda
Classi: {0: 'LABEL_0', 1: 'LABEL_1'}


## 4. Funzioni per il calcolo delle attribuzioni IG

Il modello classifica l'intera frase (semplice / complessa).  
IG ci dice quali token hanno contribuito maggiormente alla classificazione come **complessa**.

Siccome BERTino usa tokenizzazione wordpiece (subword), aggreghiamo le attribuzioni dei subword
sul token spaCy originale tramite `word_ids()`.

In [4]:
def build_lig(model, device):
    """Costruisce il LayerIntegratedGradients sul layer di embedding del modello."""

    emb_layer = model.get_input_embeddings()

    # def forward_fn(input_ids, attention_mask):
    #     # Ritorna la probabilità della classe "complessa" (indice 1)
    #     logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    #     return torch.softmax(logits, dim=-1)[:, 1]
    
    def forward_fn(input_ids, attention_mask):
        # Ritorna il logit della classe "complessa" (indice 1)
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        return logits[:, 1]

    return LayerIntegratedGradients(forward_fn, emb_layer)

In [5]:
def compute_ig_scores(example, tok, lig, n_steps=50, device="cuda"):
    """
    Calcola le attribuzioni IG per una singola frase.
    Ritorna un array numpy di shape [n_token_spaCy] con i punteggi aggregati.
    """
    # Tokenizzazione con is_split_into_words=True: i token spaCy sono già splittati
    enc = tok(example["tokens"], is_split_into_words=True, return_tensors="pt",
              truncation=True, max_length=128)
    input_ids      = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)
    word_ids       = enc.word_ids(0)   # mappa posizione subword → indice token spaCy

    # Baseline: stessa struttura dell'input ma con [PAD] al posto dei token reali
    ref = input_ids.clone()
    special_mask = tok.get_special_tokens_mask(input_ids[0].tolist(),
                                               already_has_special_tokens=True)
    for k, is_special in enumerate(special_mask):
        if not is_special:
            ref[0, k] = tok.pad_token_id

    # Calcolo attribuzioni IG sull'embedding layer
    attr = lig.attribute(
        inputs=input_ids,
        baselines=ref,
        additional_forward_args=(attention_mask,),
        n_steps=n_steps,
    )
    # Somma sulla dimensione hidden → un valore per subword
    attr = attr.sum(dim=-1).squeeze(0).detach().cpu()   # shape: [seq_len]

    # Aggregazione subword → token spaCy (somma)
    n = len(example["tokens"])
    agg = np.zeros(n)
    for k, wid in enumerate(word_ids):
        if wid is not None and wid < n:
            agg[wid] += float(attr[k])

    return agg


lig = build_lig(model, DEVICE)
print("LIG pronto.")

LIG pronto.


#### Vediamo cosa fanno i singoli pezzettini per capire

In [6]:
example = eval_set[45]
# Tokenizzazione con is_split_into_words=True: i token spaCy sono già splittati
enc = tok(example["tokens"], is_split_into_words=True, return_tensors="pt",
              truncation=True, max_length=128)

In [7]:
enc

{'input_ids': tensor([[  102,   468,   213, 21895,   213,  9473,  1156,   176,   106,  1481,
          1253, 20746,   701,  1156,   183,   214, 12690,   119,  2017,   151,
           185,  1492,   119,  9390,  1508,   277,   105,  1481, 29967,  2167,
           188,   152,   712,  7051,   687,   103]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [8]:
device = 'cuda'
input_ids      = enc["input_ids"].to(device)
attention_mask = enc["attention_mask"].to(device)
word_ids       = enc.word_ids(0)   # mappa posizione subword → indice token spaCy

In [9]:
input_ids

tensor([[  102,   468,   213, 21895,   213,  9473,  1156,   176,   106,  1481,
          1253, 20746,   701,  1156,   183,   214, 12690,   119,  2017,   151,
           185,  1492,   119,  9390,  1508,   277,   105,  1481, 29967,  2167,
           188,   152,   712,  7051,   687,   103]], device='cuda:0')

In [10]:
attention_mask

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')

In [11]:
word_ids

[None,
 0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 7,
 8,
 9,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 21,
 22,
 22,
 23,
 24,
 24,
 25,
 26,
 27,
 28,
 None]

In [12]:
  # Baseline: stessa struttura dell'input ma con [PAD] al posto dei token reali
ref = input_ids.clone()
special_mask = tok.get_special_tokens_mask(input_ids[0].tolist(),
                                               already_has_special_tokens=True)
for k, is_special in enumerate(special_mask):
        if not is_special:
            ref[0, k] = tok.pad_token_id

In [13]:
# Calcolo attribuzioni IG sull'embedding layer
attr = lig.attribute(
        inputs=input_ids,
        baselines=ref,
        additional_forward_args=(attention_mask,),
        n_steps=50,
    )

In [14]:
attr

tensor([[[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -0.0000e+00,
          -0.0000e+00, -0.0000e+00],
         [ 5.1454e-03,  6.3257e-03, -1.8189e-03,  ...,  2.1852e-04,
           2.8592e-04, -2.8026e-04],
         [-4.9075e-04,  6.7030e-04,  2.8323e-05,  ...,  4.7918e-05,
           6.2822e-04,  1.5667e-03],
         ...,
         [-2.2522e-03,  7.8350e-04,  4.9275e-04,  ..., -1.2031e-04,
           2.6323e-03, -1.4837e-03],
         [-1.8385e-03, -3.0285e-04, -6.5178e-04,  ..., -6.9202e-04,
          -3.7859e-04,  1.9832e-03],
         [ 0.0000e+00, -0.0000e+00, -0.0000e+00,  ...,  0.0000e+00,
          -0.0000e+00,  0.0000e+00]]], device='cuda:0')

In [15]:
attr.shape

torch.Size([1, 36, 768])

In [16]:
# Somma sulla dimensione hidden → un valore per subword
attr = attr.sum(dim=-1).squeeze(0).detach().cpu()   # shape: [seq_len]

In [17]:
attr

tensor([ 0.0000e+00,  1.2129e-01,  1.3452e-01, -5.3609e-02,  2.4727e-01,
        -1.3558e-01, -8.5761e-02, -1.8209e-01, -9.3997e-01,  7.5211e-03,
        -1.1467e-01,  6.1116e-02,  5.8962e-02,  1.1668e-01,  2.9573e-01,
        -2.6703e-01,  1.0359e-02, -2.9838e-02,  1.0635e-01, -3.1374e-02,
        -1.6589e-02,  1.3952e-01, -2.6109e-02,  1.8434e-02, -1.1879e-02,
         6.1880e-04,  2.0267e-01,  2.2369e-02, -7.6750e-02,  3.0785e-01,
         8.3224e-02, -1.5735e-02, -5.0209e-02,  5.3407e-02,  7.4999e-01,
         0.0000e+00])

In [18]:
# Aggregazione subword → token spaCy (somma)
n = len(example["tokens"])
agg = np.zeros(n)

for k, wid in enumerate(word_ids):
        if wid is not None and wid < n:
            agg[wid] += float(attr[k])

In [19]:
agg

array([ 0.12129062,  0.13452341, -0.05360877,  0.24726611, -0.13557675,
       -0.08576114, -0.18208551, -0.93244792, -0.11467444,  0.12007836,
        0.11667876,  0.29573321, -0.26703352,  0.01035879, -0.02983825,
        0.10634738, -0.03137435, -0.01658921,  0.13951635, -0.02610857,
        0.01843371, -0.01126017,  0.22504136, -0.07674987,  0.39107485,
       -0.01573506, -0.05020934,  0.05340719,  0.74998999])

In [20]:
len(agg)

29

In [21]:
def compute_ig_scores(example, tok, lig, n_steps=50, device="cuda"):
    """
    Calcola le attribuzioni IG per una singola frase.
    Ritorna un array numpy di shape [n_token_spaCy] con i punteggi aggregati.
    """
    # Tokenizzazione con is_split_into_words=True: i token spaCy sono già splittati
    enc = tok(example["tokens"], is_split_into_words=True, return_tensors="pt",
              truncation=True, max_length=128)
    
    input_ids      = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)
    word_ids       = enc.word_ids(0)   # mappa posizione subword → indice token spaCy

    # Baseline: stessa struttura dell'input ma con [PAD] al posto dei token reali
    ref = input_ids.clone()
    special_mask = tok.get_special_tokens_mask(input_ids[0].tolist(),
                                               already_has_special_tokens=True)
    for k, is_special in enumerate(special_mask):
        if not is_special:
            ref[0, k] = tok.pad_token_id

    # Calcolo attribuzioni IG sull'embedding layer
    attr = lig.attribute(
        inputs=input_ids,
        baselines=ref,
        additional_forward_args=(attention_mask,),
        n_steps=n_steps,
    )
    # Somma sulla dimensione hidden → un valore per subword
    attr = attr.sum(dim=-1).squeeze(0).detach().cpu()   # shape: [seq_len]

    # Aggregazione subword → token spaCy (somma)
    n = len(example["tokens"])
    agg = np.zeros(n)
    for k, wid in enumerate(word_ids):
        if wid is not None and wid < n:
            agg[wid] += float(attr[k])

    return agg


lig = build_lig(model, DEVICE)
print("LIG pronto.")

LIG pronto.


## 5. Predizione del modello su una frase

Prima di visualizzare le attribuzioni, verifichiamo che il modello classifichi correttamente la frase.

In [27]:
eval_df = pd.DataFrame(eval_set)
eval_df.head(5)

,original,idx,tokens,lemmas,pos,silver,tipo,simp_tokens
0,"Prodotto dalla BBC, il film esce solo nel 1998...",10,"[Prodotto, dalla, BBC, ,, il, film, esce, solo...","[prodotto, da il, bbc, ,, il, film, uscire, so...","[VERB, ADP, PROPN, PUNCT, DET, NOUN, VERB, ADV...","[1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, ...","[sub, sub, sub, keep, keep, keep, keep, sub, k...","[Nel, 1998, ,, il, film, è, stato, rilasciato,..."
1,"ad aprile per la fiera del bestiame, a giugno ...",20,"[ad, aprile, per, la, fiera, del, bestiame, ,,...","[a, aprile, per, il, fiera, di il, bestiame, ,...","[ADP, NOUN, ADP, DET, NOUN, ADP, NOUN, PUNCT, ...","[1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[del, keep, keep, keep, keep, keep, keep, keep...","[aprile, per, la, fiera, del, bestiame, ,, giu..."
2,La popolarità e il prestigio di Marco Antonio ...,22,"[La, popolarità, e, il, prestigio, di, Marco, ...","[il, popolarità, e, il, prestigio, di, marco, ...","[DET, NOUN, CCONJ, DET, NOUN, ADP, PROPN, PROP...","[0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, ...","[keep, keep, keep, keep, sub, keep, keep, keep...","[Questo, fatto, non, ha, influenzato, la, popo..."
3,Pur di riconquistare la ex ragazza Elaine ora ...,42,"[Pur, di, riconquistare, la, ex, ragazza, Elai...","[pure, di, riconquistare, il, ex, ragazza, ela...","[ADV, ADP, VERB, DET, ADJ, NOUN, PROPN, ADV, V...","[1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, ...","[sub, sub, keep, del, del, del, keep, del, del...","[Per, riconquistare, Elaine, ,, Striker, ha, p..."
4,"Cabrera firmò, all'età di 17 anni, con i New Y...",80,"[Cabrera, firmò, ,, all', età, di, 17, anni, ,...","[cabrera, firmare, ,, a il, età, di, 17, anno,...","[PROPN, VERB, PUNCT, ADP, NOUN, ADP, NUM, NOUN...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...","[del, keep, keep, keep, keep, keep, keep, keep...","[All', età, di, 17, anni, ,, firmò, con, i, Ne..."


In [28]:
def predict_sentence(ex, tok, model, device):
    """Ritorna la probabilità P(complessa) e la classe predetta."""

    enc = tok(ex["tokens"], is_split_into_words=True, return_tensors="pt",
              truncation=True, max_length=128).to(device)
    
    with torch.no_grad():
        logits = model(**enc).logits
    probs = torch.softmax(logits, dim=-1)[0]
    pred_class = int(probs.argmax())
    label_map = {0: "semplice", 1: "complessa"}
    
    return {
        "pred_class": pred_class,
        "label":      label_map[pred_class],
        "p_semplice": float(probs[0]),
        "p_complessa": float(probs[1]),
    }

# Test su un esempio
ex0 = eval_set[0]
pred = predict_sentence(ex0, tok, model, DEVICE)
print(f"Frase: {ex0['original'][:80]}...")
print(f"Predizione: {pred['label']}  (P_complessa={pred['p_complessa']:.3f})")

Frase: Prodotto dalla BBC, il film esce solo nel 1998 ed ottiene numerosi riconosciment...
Predizione: complessa  (P_complessa=0.972)


## 6. Visualizzazione IG — singola frase

Modifica `idx` per scegliere quale frase analizzare.  
Il colore di sfondo codifica l'attribuzione:
- **verde** → spinge verso "complessa"
- **rosso** → spinge verso "semplice"
- intensità = quanto fortemente

In [30]:
def visualizza_singola(idx_esempio, eval_set, tok, model, lig, n_steps=50, device="cuda"):

    ex   = eval_set[idx_esempio]
    pred = predict_sentence(ex, tok, model, device)
    agg  = compute_ig_scores(ex, tok, lig, n_steps=n_steps, device=device)

    agg_t = torch.tensor(agg, dtype=torch.float)
    norm  = float(agg_t.norm()) or 1.0

    record = viz.VisualizationDataRecord(
        word_attributions = agg_t / norm,
        pred_prob         = pred["p_complessa"],
        pred_class        = pred["label"],
        true_class        = "complessa" if any(ex["silver"]) else "semplice",
        attr_class        = "complessa",
        attr_score        = float(agg_t.sum()),
        raw_input_ids     = ex["tokens"],
        convergence_score = 0.0,
    )

    print(f"\nFrase #{idx_esempio}:")
    print(f"  Originale:     {ex['original'][:100]}")
    print(f"  Semplificata:  {' '.join(ex['simp_tokens'])[:100]}")
    print(f"  Predizione:    {pred['label']}  (P_complessa={pred['p_complessa']:.3f})")
    print(f"  Silver labels: {sum(ex['silver'])}/{len(ex['silver'])} token complessi")
    return viz.visualize_text([record])

# Cambia questo numero per esplorare esempi diversi
IDX = 478
html_out = visualizza_singola(IDX, eval_set, tok, model, lig, n_steps=50, device=DEVICE)
#display(html_out)


Frase #478:
  Originale:     Il veicolo in parola è una macchina assai insolita, con 3 uomini nella parte anteriore, pilota, capo
  Semplificata:  Questo veicolo è una macchina insolita , con 3 uomini nella parte anteriore , pilota , capocarro e m
  Predizione:    complessa  (P_complessa=0.938)
  Silver labels: 4/23 token complessi


True Label,Predicted Label,Attribution Label,Attribution Score,Word Importance
complessa,complessa (0.94),complessa,3.22,"Il veicolo in parola è una macchina assai insolita , con 3 uomini nella parte anteriore , pilota , capocarro e mitragliere ."


## 7. Visualizzazione IG — confronto tra più frasi

Utile per confrontare frasi con diversa complessità o diversi pattern linguistici.

In [31]:
def visualizza_multipla(indici, eval_set, tok, model, lig, n_steps=50, device="cuda"):
    records = []
    for i in indici:
        ex   = eval_set[i]
        pred = predict_sentence(ex, tok, model, device)
        agg  = compute_ig_scores(ex, tok, lig, n_steps=n_steps, device=device)

        agg_t = torch.tensor(agg, dtype=torch.float)
        norm  = float(agg_t.norm()) or 1.0

        records.append(viz.VisualizationDataRecord(
            word_attributions = agg_t / norm,
            pred_prob         = pred["p_complessa"],
            pred_class        = pred["label"],
            true_class        = "complessa" if any(ex["silver"]) else "semplice",
            attr_class        = "complessa",
            attr_score        = float(agg_t.sum()),
            raw_input_ids     = ex["tokens"],
            convergence_score = 0.0,
        ))

        p_c = pred["p_complessa"]
        n_c = sum(ex["silver"])
        print(f"  [{i}] P_complessa={p_c:.2f} | {n_c}/{len(ex['silver'])} tok silver | {ex['original'][:70]}")

    return viz.visualize_text(records)

# Visualizza 4 esempi a scelta
INDICI = [0, 1, 2, 3]
html_multi = visualizza_multipla(INDICI, eval_set, tok, model, lig, n_steps=50, device=DEVICE)
#display(html_multi)

  [0] P_complessa=0.97 | 11/29 tok silver | Prodotto dalla BBC, il film esce solo nel 1998 ed ottiene numerosi ric
  [1] P_complessa=0.95 | 14/44 tok silver | ad aprile per la fiera del bestiame, a giugno per la compravendita del
  [2] P_complessa=1.00 | 6/17 tok silver | La popolarità e il prestigio di Marco Antonio peraltro non soffrirono 
  [3] P_complessa=0.99 | 20/30 tok silver | Pur di riconquistare la ex ragazza Elaine ora impiegata come hostess, 


## 9. Confronto visivo: silver labels vs attribuzioni IG

Ogni frase viene mostrata due volte:
- **Riga SILVER**: sfondo rosa = token cancellato nella semplificazione · azzurro = token sostituito · grigio = mantenuto
- **Riga IG**: intensità del colore = forza dell'attribuzione verso «complessa» · bordo scuro = token nel top-k selezionato da IG

Le metriche P/R/F1 in fondo indicano quanto IG e silver labels concordano su *questo* esempio.

In [32]:
def _ig_color(score, max_pos, max_neg):
    """Arancio/rosso per attributions positive, azzurro per negative."""
    if score >= 0:
        t = min(score / max_pos, 1.0)
        r, g, b = 255, int(255 * (1 - t * 0.85)), int(255 * (1 - t))
    else:
        t = min(-score / max_neg, 1.0)
        r, g, b = int(255 * (1 - t)), int(255 * (1 - t * 0.4)), 255
    return f"rgb({r},{g},{b})"


def html_confronto(ex, agg, top_frac=0.3):
    """
    Ritorna una stringa HTML con:
    - riga 1: token colorati secondo la silver label (cancellazione/sostituzione/mantenuto)
    - riga 2: token colorati per intensità IG, bordo scuro sui token nel top-k
    - riga 3: metriche P/R/F1/TP/FP/FN dell'esempio
    """
    n     = len(ex["tokens"])
    k_top = max(1, int(round(top_frac * n)))
    top_ig = set(np.argsort(agg)[-k_top:])

    max_pos = max(float(agg.max()), 1e-9)
    max_neg = max(float(-agg.min()), 1e-9)

    # riga silver
    silver_spans = []
    for token, tipo in zip(ex["tokens"], ex["tipo"]):
        bg = {"del": "#fbb6ce", "sub": "#bfdbfe"}.get(tipo, "#f3f4f6")
        silver_spans.append(
            f'<span style="background:{bg};padding:3px 6px;margin:2px;'
            f'border-radius:4px;display:inline-block">{token}</span>'
        )

    # riga IG
    ig_spans = []
    for i, (token, score) in enumerate(zip(ex["tokens"], agg)):
        bg     = _ig_color(score, max_pos, max_neg)
        border = "2px solid #7c2d12" if i in top_ig else "2px solid transparent"
        weight = "bold" if i in top_ig else "normal"
        ig_spans.append(
            f'<span style="background:{bg};padding:3px 6px;margin:2px;'
            f'border-radius:4px;border:{border};font-weight:{weight};'
            f'display:inline-block">{token}</span>'
        )

    # metriche
    ig_labels = [int(i in top_ig) for i in range(n)]
    tp = sum(s == 1 and p == 1 for s, p in zip(ex["silver"], ig_labels))
    fp = sum(s == 0 and p == 1 for s, p in zip(ex["silver"], ig_labels))
    fn = sum(s == 1 and p == 0 for s, p in zip(ex["silver"], ig_labels))
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    pct  = round(top_frac * 100)

    return f"""
    <div style="font-family:sans-serif;margin:16px 0;padding:14px 16px;
                border:1px solid #e5e7eb;border-radius:8px;background:#fff">
      <div style="font-size:.82em;color:#6b7280;margin-bottom:10px">
        <b>Originale:</b> {ex['original'][:130]}{'...' if len(ex['original']) > 130 else ''}
      </div>

      <div style="font-size:.78em;font-weight:bold;color:#374151;margin:6px 0 2px">
        SILVER LABELS
        <span style="font-weight:normal;color:#9ca3af;margin-left:6px">
          &#9632; <span style="color:#fbb6ce">rosa</span> = cancellato &nbsp;
          &#9632; <span style="color:#bfdbfe">azzurro</span> = sostituito &nbsp;
          &#9632; <span style="color:#f3f4f6">grigio</span> = mantenuto
        </span>
      </div>
      <div style="line-height:2.4;margin-bottom:10px">{"".join(silver_spans)}</div>

      <div style="font-size:.78em;font-weight:bold;color:#374151;margin:6px 0 2px">
        INTEGRATED GRADIENTS &nbsp;
        <span style="font-weight:normal;color:#9ca3af">
          arancio/rosso = spinge verso «complessa» &nbsp;|&nbsp;
          azzurro = spinge verso «semplice» &nbsp;|&nbsp;
          bordo scuro = top {pct}%
        </span>
      </div>
      <div style="line-height:2.4;margin-bottom:10px">{"".join(ig_spans)}</div>

      <div style="font-size:.8em;color:#1e293b;background:#f8fafc;
                  padding:7px 12px;border-radius:4px;border-left:3px solid #6366f1">
        <b>P={prec:.2f} &nbsp; R={rec:.2f} &nbsp; F1={f1:.2f}</b>
        &nbsp;&nbsp;|&nbsp;&nbsp; TP={tp} &nbsp; FP={fp} &nbsp; FN={fn}
        &nbsp;&nbsp;|&nbsp;&nbsp; {sum(ex['silver'])}/{n} token complessi (silver)
        &nbsp; · &nbsp; {k_top}/{n} selezionati da IG
      </div>
    </div>
    """


def visualizza_confronto(indici, eval_set, tok, model, lig,
                         top_frac=0.3, n_steps=50, device="cuda"):
    """Mostra il confronto silver vs IG per gli esempi in `indici`."""
    blocchi = []
    for i in indici:
        ex  = eval_set[i]
        agg = compute_ig_scores(ex, tok, lig, n_steps=n_steps, device=device)
        blocchi.append(html_confronto(ex, agg, top_frac=top_frac))
    display(HTML("".join(blocchi)))


# Cambia gli indici per esplorare esempi diversi
visualizza_confronto([0, 1, 2], eval_set, tok, model, lig, top_frac=0.3, device=DEVICE)

## 11. Visualizzazione interattiva — classe XAI

Interfaccia alternativa che accetta una frase grezza (stringa) e produce:
- barre di probabilità Simple / Complex
- testo color-coded per attribuzione IG
- DataFrame con i top-k token più influenti

In [33]:
class XAI:
    def __init__(self, text_, label_, tokenizer_, model_, device_):
        self.text = text_
        self.label = label_
        self.tokenizer = tokenizer_
        self.model = model_
        self.device = device_
        self.input_ids = None
        self.ref_input_ids = None
        self.tokens = None

    def construct_input_ref(self):
        text_ids = self.tokenizer.encode(self.text, add_special_tokens=False)
        input_ids = ([self.tokenizer.cls_token_id] +
                     text_ids +
                     [self.tokenizer.sep_token_id])
        ref_input_ids = ([self.tokenizer.cls_token_id] +
                         [self.tokenizer.pad_token_id] * len(text_ids) +
                         [self.tokenizer.sep_token_id])
        self.input_ids = torch.tensor([input_ids], device=self.device)
        self.ref_input_ids = torch.tensor([ref_input_ids], device=self.device)
        return self.input_ids, self.ref_input_ids

    def custom_forward(self, inputs):
        # Ritorna P(complessa) come tensore [batch] — scalare per Captum
        logits = self.model(inputs)[0]
        return torch.softmax(logits, dim=-1)[:, 1]

    def compute_attributions(self):
        self.input_ids, self.ref_input_ids = self.construct_input_ref()
        self.tokens = self.tokenizer.convert_ids_to_tokens(self.input_ids[0])

        # get_input_embeddings() funziona con qualsiasi architettura (BERT, DistilBERT, ecc.)
        lig = LayerIntegratedGradients(self.custom_forward, self.model.get_input_embeddings())
        attributions, delta = lig.attribute(
            inputs=self.input_ids,
            baselines=self.ref_input_ids,
            n_steps=500,
            internal_batch_size=3,
            return_convergence_delta=True,
        )
        attributions = attributions.sum(dim=-1).squeeze()
        normalized_attributions = attributions / torch.norm(attributions)
        return normalized_attributions, self.tokens

    def predict_probabilities(self):
        with torch.no_grad():
            logits = self.model(self.input_ids)[0]
        return torch.softmax(logits, dim=-1)[0].tolist()  # [p_simple, p_complex]

    def get_topk_attributed_tokens(self, attrs, k=5):
        attrs = attrs.cpu()
        values, indices = torch.topk(attrs, k)
        return pd.DataFrame({
            'Word':        [self.tokens[idx] for idx in indices],
            'Index':       indices.cpu().numpy(),
            'Attribution': values.cpu().numpy(),
        })

    def generate_html(self, attributions, tokens, probabilities):
        token_html = ""
        for token, score in zip(tokens, attributions):
            score = float(score)
            color = (f"rgba(255, 0, 0, {abs(score):.3f})" if score < 0
                     else f"rgba(0, 0, 255, {abs(score):.3f})")
            token_html += f"<span style='background-color:{color};padding:2px 4px;margin:1px;border-radius:3px'>{token}</span> "

        return f"""
        <div style="font-family:sans-serif;margin:16px 0;padding:14px 16px;
                    border:1px solid #e5e7eb;border-radius:8px;background:#fff">
            <h4 style="margin-top:0">Prediction Probabilities</h4>
            <div style="margin-bottom:12px">
                <div>Simple</div>
                <div style="width:100%;height:20px;background:#ddd;border-radius:5px;margin:5px 0">
                    <div style="width:{probabilities[0]*100:.1f}%;height:100%;background:blue;border-radius:5px"></div>
                </div>
                <p style="margin:2px 0">Probability: {probabilities[0]:.3f}</p>

                <div>Complex</div>
                <div style="width:100%;height:20px;background:#ddd;border-radius:5px;margin:5px 0">
                    <div style="width:{probabilities[1]*100:.1f}%;height:100%;background:orange;border-radius:5px"></div>
                </div>
                <p style="margin:2px 0">Probability: {probabilities[1]:.3f}</p>
            </div>

            <h4>Text with Highlighted Words</h4>
            <p style="line-height:2.2">{token_html}</p>
            <p style="font-size:.78em;color:#6b7280">
                <span style="background:rgba(0,0,255,0.4);padding:2px 6px">blu</span> = spinge verso «complessa» &nbsp;
                <span style="background:rgba(255,0,0,0.4);padding:2px 6px">rosso</span> = spinge verso «semplice»
            </p>
        </div>
        """


def analyze_and_display(sentence, label, tokenizer, model, device, top_k=None):
    model.to(device)
    xai_instance = XAI(
        text_=sentence,
        label_=label,
        tokenizer_=tokenizer,
        model_=model,
        device_=device,
    )
    attributions, tokens = xai_instance.compute_attributions()
    probabilities = xai_instance.predict_probabilities()
    display(HTML(xai_instance.generate_html(attributions, tokens, probabilities)))

    if top_k is None:
        top_k = len(sentence.split())
    display(xai_instance.get_topk_attributed_tokens(attributions, k=top_k))


print("XAI class e analyze_and_display pronti.")

XAI class e analyze_and_display pronti.


In [38]:
# Demo: analizza una frase arbitraria
sentence = eval_set[6]["original"]
label = 1

analyze_and_display(sentence, label, tok, model, DEVICE, top_k=5)

,Word,Index,Attribution
0,(,4,0.625671
1,.,25,0.525577
2,",",18,0.283211
3,o,5,0.254985
4,nacque,12,0.188751
